Started 6.52

In [ ]:
import jax
from flax import nnx
from jax import numpy as jnp
from flax.typing import Shape

def softmax_cross_attention(q: jax.Array, k: jax.Array, v: jax.Array, *, is_causal = False, mask: jax.Array | None = None):
    T_q, T_k = q.shape[-3], k.shape[-3]
    d_head = q.shape[-1]

    _mask = jnp.tri(T_q, T_k, T_k - T_q, dtype=bool) if is_causal else jnp.ones((T_q, T_k), dtype=bool)
    _mask = _mask & mask if mask is not None else _mask

    attn_scores: jax.Array = jnp.einsum("...qhd,...khd->...hqk", q, k) / d_head ** 0.5
    attn_scores = jnp.where(_mask, attn_scores, -jnp.inf)

    attn_weights = nnx.softmax(attn_scores, axis=-1)

    return jnp.einsum("...hqk,...khd->...qhd", attn_weights, v)

class MultiHeadAttention(nnx.Module):
    def __init__(self, d_model: int, d_head: int, *, rngs: nnx.Rngs):
        assert d_model % d_head == 0
        n_head = d_model // d_head

        projection = lambda : nnx.Einsum(
            "...e,ehd->...hd",
            (d_model, n_head, d_head),
            (n_head, d_head),
            rngs=rngs
        )

        self.q = projection()
        self.k = projection()
        self.v = projection()

        self.o = nnx.Einsum(
            "...hd,hde->...e",
            (n_head, d_head, d_model),
            (d_model,),
            rngs=rngs
        )

        self.ck: None | nnx.Cache[jax.Array] = nnx.data(None) # Why does this have to be nnx.data(None), does not need to be tracked until replaced with the array..?
        self.cv: None | nnx.Cache[jax.Array] = nnx.data(None)
        self.ci: None | nnx.Cache[jax.Array] = nnx.data(None)
        
    def init_cache(self, shape: Shape):
        self.ck = nnx.Cache(jnp.zeros(shape))
        self.cv = nnx.Cache(jnp.zeros(shape))
        self.ci = nnx.Cache(jnp.array(0))

    def __call__(self, X: jax.Array, memory: jax.Array | None = None, *, is_causal = False, mask: jax.Array | None = None, decode = False):
        if memory is None:
            memory = X

        q = self.q(X)
        k = self.k(memory)
        v = self.v(memory)

        if decode:
            assert self.ck is not None
            assert self.cv is not None
            assert self.ci is not None
            assert k.shape[-3] == 1 # only process one new token at a time
            assert is_causal == False

            (*batch, key_len, _n_head, _d_head) = self.ck[...].shape

            zero = jnp.array(0)
            indices = (zero,) * len(batch) + (self.ci[...], zero, zero)
            k = jax.lax.dynamic_update_slice(self.ck[...], k, indices)
            v = jax.lax.dynamic_update_slice(self.cv[...], v, indices)

            self.ck[...] = k
            self.cv[...] = v
            self.ci[...] += 1

            # batch...hqk
            # 1,1,1...1,1,k
            cache_mask = jnp.broadcast_to(
                jnp.arange(key_len) < self.ci[...],
                (1,) * len(batch) + (1, 1, key_len),
            )
            mask = cache_mask if mask is None else mask & cache_mask

        y = softmax_cross_attention(q, k, v, is_causal=is_causal, mask=mask)

        return self.o(y)

In [ ]:
from einops import rearrange

class ViT_Block(nnx.Module):
    def __init__(self, d_model, d_head, d_ffn, *, rngs: nnx.Rngs):
        self.mha = MultiHeadAttention(d_model, d_head, rngs=rngs)
        self.mlp = nnx.Sequential(
            nnx.Linear(d_model, d_ffn, rngs=rngs),
            nnx.gelu,
            nnx.Linear(d_ffn, d_model, rngs=rngs)
        )
        self.norm1 = nnx.LayerNorm(d_model, rngs=rngs)
        self.norm2 = nnx.LayerNorm(d_model, rngs=rngs)

    def __call__(self, X: jax.Array):
        y = X + self.mha(self.norm1(X))
        y = y + self.mlp(self.norm2(y))

        return y

class ViT(nnx.Module):
    """
    Takes NHWC
    Produces NHWD
    """

    def __init__(self, d_model: int, d_head: int, d_ffn: int, layers: int = 8, *, 
                patch_size: Shape = (14, 14), 
                strides: Shape = (14, 14),
                rngs: nnx.Rngs,
                in_features: int = 3
            ):

        # NHWC -> NHWD
        self.patch_embed = nnx.Conv(in_features, d_model, patch_size, strides, rngs=rngs, padding="VALID")
        self.layers = nnx.Sequential(*(ViT_Block(d_model, d_head, d_ffn, rngs=rngs) for _ in range(layers)))

    def __call__(self, X: jax.Array):
        # I KNOW there's no RoPE, will figure out later
        y = self.patch_embed(X)
        h = y.shape[-3]
        y = rearrange(y, "... h w d -> ... (h w) d")

        # using pre-norm
        y = self.layers(y)

        y = rearrange(y, "... (h w) d -> ... h w d", h=h)

        return y

In [1]:
(1,) * 2

(1, 1)

In [ ]:
import jax.numpy as jnp
from flax import nnx

In [ ]:
class Test(nnx.Module):
    def __init__(self):
        self.cache = None

m = Test()
m.cache = nnx.Cache(jnp.zeros(10)) # type:ignore
print(m.cache)

ValueError: Cannot assign data value of type '<class 'flax.nnx.variablelib.Cache'>' to static attribute 'cache' of Pytree type '<class '__main__.Test'>'. To override the status explicitly wrap the value with nnx.data on assignment:

  _.cache = nnx.data(...)



In [ ]:
class Test(nnx.Module):
    def __init__(self):
        self.cache = None

m = Test()
m.cache = nnx.data(nnx.Cache(jnp.zeros(10))) # type:ignore
print(m.cache)

Cache( # 10 (40 B)
  value=Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)
)


In [ ]:
class Test(nnx.Module):
    def __init__(self):
        self.cache = nnx.Cache(None)

m = Test()
m.cache = nnx.Cache(jnp.zeros(10)) # type:ignore
print(m.cache)

Cache( # 10 (40 B)
  value=Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)
)
